In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

In [ ]:
# 9. Flash Attention 核心：分块 + online softmax，不物化完整 [T,S] 矩阵
def flash_attention_toy(q, k, v, block_size=4):
    # 教学版：q/k/v [B,T,D]，非 causal
    B, T, D = q.shape
    assert q.shape == k.shape and v.shape == q.shape
    output = torch.empty_like(q)
    for i in range(0, T, block_size):
        qi = q[:, i:i + block_size]
        m = torch.full((*qi.shape[:2], 1), -torch.inf, device=q.device, dtype=q.dtype)
        l = torch.zeros_like(m)
        acc = torch.zeros_like(qi)
        for j in range(0, T, block_size):
            kj, vj = k[:, j:j + block_size], v[:, j:j + block_size]
            scores = qi @ kj.transpose(-2, -1) / math.sqrt(D)
            block_max = scores.max(dim=-1, keepdim=True).values
            new_m = torch.maximum(m, block_max)
            correction = torch.exp(m - new_m)
            p = torch.exp(scores - new_m)
            acc = acc * correction + p @ vj
            l = l * correction + p.sum(dim=-1, keepdim=True)
            m = new_m
        output[:, i:i + block_size] = acc / l
    return output

q = torch.randn(2, 9, 8, device=device)
k = torch.randn(2, 9, 8, device=device)
v = torch.randn(2, 9, 8, device=device)
ref = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(8), dim=-1) @ v
actual = flash_attention_toy(q, k, v, block_size=4)
assert torch.allclose(ref, actual, atol=1e-5), (ref - actual).abs().max()